# 01 - изучить модель SmolVLA 

Цель notebook: пока просто скачать + проверить совместимость с датасетом

Скачаем `lerobot/smolvla_base` у неё pretrain модули это 

In [16]:
from pathlib import Path
from huggingface_hub import snapshot_download
from lerobot.configs import PreTrainedConfig
from pprint import pprint
import torch

ROOT_DIR = Path.cwd().resolve().parent

BASE_MODEL_DIR = ROOT_DIR / "models" / "smolvla_base"

In [11]:
# snapshot_download(
#     repo_id="lerobot/smolvla_base",
#     repo_type="model",
#     local_dir=BASE_MODEL_DIR,
# )

## Инженерные проблемы, с которыми столкнулся 

Я понял, что надо переименовывать так как в dataset от nvidia они называются по другому:\
```
rename_map='{
    "observation.images.image":"observation.images.camera1", 
    "observation.images.wrist_image":"observation.images.camera2", 
    "observation.images.image2":"observation.images.camera2" 
}'
``` 
`wrist_image` ещё заменяется на image2 [тут](https://github.com/huggingface/lerobot/blob/22bd7a2f489b367d8df42de803b1e8c4ca63a3f9/src/lerobot/envs/utils.py#L80) 

Нормировка картинок в libero в [0, 1] (я проверил глазками на всякий случай) в конце ноутбука 00 \
И потом она идёт в smolvla в [lerobot](https://github.com/huggingface/lerobot/blob/22bd7a2f489b367d8df42de803b1e8c4ca63a3f9/src/lerobot/policies/smolvla/modeling_smolvla.py#L360)
`img = img * 2.0 - 1.0`

И также smolvla_base имеет размерность action [6] при инициализации \

А данные в `nvidia/LIBERO_LeRobot_v3` у action [7] (как я понял подаётся ещё gripper) \
И но так как происходит [padding](https://github.com/huggingface/lerobot/blob/22bd7a2f489b367d8df42de803b1e8c4ca63a3f9/src/lerobot/policies/smolvla/modeling_smolvla.py#L417) до [max_action_dim: int = 32](https://github.com/huggingface/lerobot/blob/22bd7a2f489b367d8df42de803b1e8c4ca63a3f9/src/lerobot/policies/smolvla/configuration_smolvla.py#L42) то у нас идёт все обучение с размерностью как `nvidia/LIBERO_LeRobot_v3`

Аналогично со state 



В окружении изменить этот файл:
`.venv/lib/python3.12/site-packages/lerobot/scripts/lerobot_eval.py`

В этом блоке поменять:
```python
action_transition = {ACTION: action}
action_transition = env_postprocessor(action_transition)
action = action_transition[ACTION]

# Convert to CPU / numpy.
action_numpy: np.ndarray = action.to("cpu").numpy()
```

Вставить между `action = action_transition[ACTION] и action_numpy:`

Чтобы стало так:
```python
action_transition = {ACTION: action}
action_transition = env_postprocessor(action_transition)
action = action_transition[ACTION]

if action.shape[-1] >= 7:
    action = action.clone()
    gripper = action[..., 6]
    action[..., 6] = 1.0 - 2.0 * (gripper > 0.5).to(action.dtype)

# Convert to CPU / numpy.
action_numpy: np.ndarray = action.to("cpu").numpy()
```

Это изменяет поведение для проведения rollout:

dataset/model gripper > 0.5 -> -1 open

dataset/model gripper <= 0.5 -> +1 close

## Про саму модель (черновик) [статья](https://arxiv.org/pdf/2506.01844)

Архитектура состоит из двух крупных частей: \
VLM -> Action Expert

VLM отвечает в основном за понимание сцены и задания, а Action Expert преобразует это представление в непрерывные управляющие воздействия. В качестве VLM используется SmolVLM-2 с SigLIP vision encoder и SmolLM2 language decoder

Интересные решения, что я подчеркнул для себя:
- VLM специально урезают (и вроде достаточно использовать половину) + VLM при обучении policy заморожен
- Action Expert - отдельный Transformer примерно на 100M параметров. Он генерирует сразу последовательность: \
    $ A_t = [a_t, a_{t+1}, ..., a_{t+n}]  \;\;\; (n = 50)$ \
    Идёт чередование слоёв: Cross Attention -> Self Attention -> Cross Attention ... \
    Cross Attention: q из action tokens; k,v из признаки(токены??) с vlm \
    Self Attention позволяет различным будущим action tokens взаимодействовать друг с другом( не очень понял как по нормальному написать ) \ 
- Flow Matching: для истинного action chunk $A_t$ формируется зашумлённая версия $A_t^\tau=(1-\tau)A_t+\tau\epsilon$, где $\epsilon\sim\mathcal{N}(0,I)$.
Модель $v_\theta(A_t^\tau,obs_t,\tau)$ по признакам VLM и зашумлённым действиям предсказывает векторное поле $u=\epsilon-A_t$, минимизируя $\mathcal{L}=\|v_\theta-u\|_2^2$.



